# PyDI Data Integration Workflow: Companies

This notebook demonstrates how PyDI is used for end-to-end data integration. We'll work with companies datasets to showcase the data integration pipeline from schema and entity matching to data fusion.

## Table of Contents
  - [Datasets](#datasets)
- [Part 1: Schema Matching and Value Normalization](#part-1-schema-matching-and-value-normalization)
- [Part 2: Data Profiling](#part-2-data-profiling)
- [Part 3: Entity Matching](#part-3-entity-matching)
  - [Step 1: Blocking](#step-1-blocking)
  - [Step 2: Blocking Evaluation](#step-2-evaluate-blocking-against-ground-truth)
  - [Step 3: Entity Matching with Comparators](#step-3-entity-matching-with-comparators)
  - [Step 4: Entity Matching Evaluation](#step-4-evaluate-matching-against-ground-truth)
- [Part 4: Data Fusion](#part-4-data-fusion)
  - [Step 1: Define Fusion Strategy](#step-1-define-fusion-strategy)
  - [Step 2: Run Fusion](#step-2-run-fusion)
  - [Step 3: Data Fusion Evaluation](#step-3-evaluate-data-fusion)
- [Part 5: Generate Reports](#part-5-generate-reports)

## Part 1: Schema Matching and Value Normalization

In [1]:
from pathlib import Path

# Paths relative to this notebook
NOTEBOOK_DIR = Path(".").resolve()
INPUT_DIR = NOTEBOOK_DIR 
OUTPUT_DIR = NOTEBOOK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
import pandas as pd
import json
from PyDI.schemamatching import SchemaTranslator
from PyDI.normalization import load_normalization_spec

/Users/aaronsteiner/Documents/GitHub/PyDI/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load Target Schema and Normalization Spec

In [3]:
# Load the JSON Schema (used for both matching and normalization)
with open(INPUT_DIR / "schemamatching" / "target_schema.json") as f:
    target_schema = json.load(f)

# Load NormalizationSpec from the schema - point to the Company definition where columns are defined
spec = load_normalization_spec(
    INPUT_DIR / "schemamatching" / "target_schema.json",
    property_path="$defs.Company"
)

# founders is an array type which gets skipped by default, so set it manually
spec.set_column("founders", output_type="list<string>")

target_columns = list(spec.columns.keys())

# Create empty target DataFrame for schema matching
df_target = pd.DataFrame(columns=target_columns)
df_target.attrs["dataset_name"] = "target_schema"

# Show column types derived from schema
pd.DataFrame([
    {"column": col, "output_type": col_spec.output_type}
    for col, col_spec in spec.columns.items()
])

,column,output_type
0,id,string
1,name,string
2,website,string
3,founded,datetime
4,country,string
5,city,string
6,industry,string
7,assets,int
8,revenue,int
9,founders,list<string>


## Step 2: Load Source Datasets

In [4]:
from PyDI.io import load_csv, load_xml, load_json
import numpy as np
dbpedia = load_csv(INPUT_DIR / "data" / "dbpedia.csv")
dbpedia.attrs["dataset_name"] = "dbpedia"
dbpedia["founders"] = dbpedia["keypeople_name"].apply(lambda x: x if x else np.NaN)
dbpedia.head(10)

,entity_uri,org_name,established,nation,headquarters,sector,keypeople_name,total_assets_val,annual_income,founders
0,http://dbpedia.org/resource/%C3%80_la_Table_de...,À la Table de Spanghero,1970-01-01,France,Castelnaudary,Meat,NaN,NaN,NaN,NaN
1,http://dbpedia.org/resource/%C3%81guas_de_Port...,�?guas de Portugal,1993-01-01,Portugal,Lisbon,NaN,NaN,NaN,NaN,NaN
2,http://dbpedia.org/resource/%C3%81nima_Estudios,�?nima Estudios,2002-01-01,Mexico,Mexico City,Animation,NaN,NaN,NaN,NaN
3,http://dbpedia.org/resource/%C3%87al%C4%B1k_En...,Çalık Enerji,1998-01-01,Turkey,Istanbul,NaN,Çalık Holding,NaN,NaN,Çalık Holding
4,http://dbpedia.org/resource/%C3%87al%C4%B1k_Ho...,Çalık Holding,1997-01-01,Turkey,Istanbul,NaN,Ahmet Çalık,8.000000e+00,2.800000e+00,Ahmet Çalık
5,http://dbpedia.org/resource/%C3%87ukurova_(con...,Çukurova (construction firm),1975-01-01,Turkey,Istanbul,NaN,NaN,NaN,NaN,NaN
6,http://dbpedia.org/resource/%C3%87ukurova_Holding,Çukurova Holding,1923-01-01,Turkey,Istanbul,Communication,NaN,NaN,NaN,NaN
7,http://dbpedia.org/resource/%C3%89lectricit%C3...,Électricité de France,1946-01-01,France,Paris,Electric utility,Marcel Paul,2.405600e+11,6.517000e+10,Marcel Paul
8,http://dbpedia.org/resource/%C3%89tranges_Libe...,Étranges Libellules,1994-01-01,France,Lyon,Video game industry,NaN,NaN,NaN,NaN
9,http://dbpedia.org/resource/%C3%96ssur,Össur,1971-01-01,Iceland,Reykjavík,Health care,NaN,6.071000e+08,3.585000e+08,NaN


In [5]:
forbes = load_csv(INPUT_DIR / "data" / "forbes.csv")
forbes.attrs["dataset_name"] = "forbes"
forbes.head()

,forbes_url,company,url,region,business_segment,asset_value,sales_figure
0,http://www.forbes.com/companies/icbc/,ICBC,http://www.forbes.com/companies/icbc/,China,Major Banks,3124900000000,148700000000
1,http://www.forbes.com/companies/china-construc...,China Construction Bank,http://www.forbes.com/companies/china-construc...,China,Regional Banks,2449500000000,121300000000
2,http://www.forbes.com/companies/agricultural-b...,Agricultural Bank of China,http://www.forbes.com/companies/agricultural-b...,China,Regional Banks,2405400000000,136400000000
3,http://www.forbes.com/companies/jpmorgan-chase/,JPMorgan Chase,http://www.forbes.com/companies/jpmorgan-chase/,United States of America,Major Banks,2435300000000,105700000000
4,http://www.forbes.com/companies/berkshire-hath...,Berkshire Hathaway,http://www.forbes.com/companies/berkshire-hath...,United States of America,Investment Services,493400000000,178800000000


In [6]:
fullcontact = load_csv(INPUT_DIR / "data" / "fullcontact.csv")
fullcontact.attrs["dataset_name"] = "fullcontact"
fullcontact.head()

,Attribute_1,Attribute_2,Attribute_3,Attribute_4,Attribute_5,Attribute_6
0,fullcontact_1,BBMG,United States,Brooklyn,Raphael Bemporad,NaN
1,fullcontact_2,CIT Group Inc (DEL),Canada,Toronto,NaN,1908-01-01
2,fullcontact_3,City & National Employment,United States,Waterloo,NaN,1957-01-01
3,fullcontact_4,Ingersoll Rand South East Asia (Pte) Ltd,Ireland,Swords,NaN,1871-01-01
4,fullcontact_5,Evonik,Germany,Essen,NaN,2007-01-01


In [7]:
# Create founders column based on keypersons_person_name and keypersons_person_title
# Ensure name/title columns are always list-like or NaN
fullcontact["keypersons_person_name"] = fullcontact["Attribute_5"].apply(
    lambda x: [x] if isinstance(x, str) else x
)
fullcontact["keypersons_person_title"] = fullcontact["Attribute_5"].apply(
    lambda x: [x] if isinstance(x, str) else x
)

def extract_founders(row):
    names = row["keypersons_person_name"]
    titles = row["keypersons_person_title"]
    if not isinstance(names, list) or not isinstance(titles, list):
        return np.nan

    founders = [
        name
        for name, title in zip(names, titles)
        if isinstance(title, str) and "founder" in title.lower()
    ]
    return founders if founders else np.nan

fullcontact["founders"] = fullcontact.apply(extract_founders, axis=1)
fullcontact.head()


,Attribute_1,Attribute_2,Attribute_3,Attribute_4,Attribute_5,Attribute_6,keypersons_person_name,keypersons_person_title,founders
0,fullcontact_1,BBMG,United States,Brooklyn,Raphael Bemporad,NaN,[Raphael Bemporad],[Raphael Bemporad],NaN
1,fullcontact_2,CIT Group Inc (DEL),Canada,Toronto,NaN,1908-01-01,NaN,NaN,NaN
2,fullcontact_3,City & National Employment,United States,Waterloo,NaN,1957-01-01,NaN,NaN,NaN
3,fullcontact_4,Ingersoll Rand South East Asia (Pte) Ltd,Ireland,Swords,NaN,1871-01-01,NaN,NaN,NaN
4,fullcontact_5,Evonik,Germany,Essen,NaN,2007-01-01,NaN,NaN,NaN


## Step 3: Manual Schema Matching

In [8]:
# Manual schema mapping for dbpedia
dbpedia_mapping = pd.DataFrame([
    {"source_dataset": "dbpedia", "source_column": "entity_uri", "target_dataset": "target_schema", "target_column": "id", "score": 1.0, "notes": "manual_mapping"},
    {"source_dataset": "dbpedia", "source_column": "org_name", "target_dataset": "target_schema", "target_column": "name", "score": 1.0, "notes": "manual_mapping"},
    {"source_dataset": "dbpedia", "source_column": "established", "target_dataset": "target_schema", "target_column": "founded", "score": 1.0, "notes": "manual_mapping"},
    {"source_dataset": "dbpedia", "source_column": "nation", "target_dataset": "target_schema", "target_column": "country", "score": 1.0, "notes": "manual_mapping"},
    {"source_dataset": "dbpedia", "source_column": "headquarters", "target_dataset": "target_schema", "target_column": "city", "score": 1.0, "notes": "manual_mapping"},
    {"source_dataset": "dbpedia", "source_column": "sector", "target_dataset": "target_schema", "target_column": "industry", "score": 1.0, "notes": "manual_mapping"},
    {"source_dataset": "dbpedia", "source_column": "total_assets_val", "target_dataset": "target_schema", "target_column": "assets", "score": 1.0, "notes": "manual_mapping"},
    {"source_dataset": "dbpedia", "source_column": "annual_income", "target_dataset": "target_schema", "target_column": "revenue", "score": 1.0, "notes": "manual_mapping"},
    {"source_dataset": "dbpedia", "source_column": "founders", "target_dataset": "target_schema", "target_column": "founders", "score": 1.0, "notes": "manual_mapping"},
])

dbpedia_mapping

,source_dataset,source_column,target_dataset,target_column,score,notes
0,dbpedia,entity_uri,target_schema,id,1.0,manual_mapping
1,dbpedia,org_name,target_schema,name,1.0,manual_mapping
2,dbpedia,established,target_schema,founded,1.0,manual_mapping
3,dbpedia,nation,target_schema,country,1.0,manual_mapping
4,dbpedia,headquarters,target_schema,city,1.0,manual_mapping
5,dbpedia,sector,target_schema,industry,1.0,manual_mapping
6,dbpedia,total_assets_val,target_schema,assets,1.0,manual_mapping
7,dbpedia,annual_income,target_schema,revenue,1.0,manual_mapping
8,dbpedia,founders,target_schema,founders,1.0,manual_mapping


In [9]:
# Manual schema mapping for forbes
forbes_mapping = pd.DataFrame([
    {"source_dataset": "forbes", "source_column": "forbes_url", "target_dataset": "target_schema", "target_column": "id", "score": 1.0, "notes": "manual_mapping"},
    {"source_dataset": "forbes", "source_column": "company", "target_dataset": "target_schema", "target_column": "name", "score": 1.0, "notes": "manual_mapping"},
    {"source_dataset": "forbes", "source_column": "region", "target_dataset": "target_schema", "target_column": "country", "score": 1.0, "notes": "manual_mapping"},
    {"source_dataset": "forbes", "source_column": "business_segment", "target_dataset": "target_schema", "target_column": "industry", "score": 1.0, "notes": "manual_mapping"},
    {"source_dataset": "forbes", "source_column": "asset_value", "target_dataset": "target_schema", "target_column": "assets", "score": 1.0, "notes": "manual_mapping"},
    {"source_dataset": "forbes", "source_column": "sales_figure", "target_dataset": "target_schema", "target_column": "revenue", "score": 1.0, "notes": "manual_mapping"},
])

forbes_mapping

,source_dataset,source_column,target_dataset,target_column,score,notes
0,forbes,forbes_url,target_schema,id,1.0,manual_mapping
1,forbes,company,target_schema,name,1.0,manual_mapping
2,forbes,region,target_schema,country,1.0,manual_mapping
3,forbes,business_segment,target_schema,industry,1.0,manual_mapping
4,forbes,asset_value,target_schema,assets,1.0,manual_mapping
5,forbes,sales_figure,target_schema,revenue,1.0,manual_mapping


In [10]:
# Manual schema mapping for fullcontact
fullcontact_mapping = pd.DataFrame([
    {"source_dataset": "fullcontact", "source_column": "Attribute_1", "target_dataset": "target_schema", "target_column": "id", "score": 1.0, "notes": "manual_mapping"},
    {"source_dataset": "fullcontact", "source_column": "Attribute_2", "target_dataset": "target_schema", "target_column": "name", "score": 1.0, "notes": "manual_mapping"},
    {"source_dataset": "fullcontact", "source_column": "Attribute_3", "target_dataset": "target_schema", "target_column": "country", "score": 1.0, "notes": "manual_mapping"},
    {"source_dataset": "fullcontact", "source_column": "Attribute_4", "target_dataset": "target_schema", "target_column": "city", "score": 1.0, "notes": "manual_mapping"},
    {"source_dataset": "fullcontact", "source_column": "Attribute_6", "target_dataset": "target_schema", "target_column": "founded", "score": 1.0, "notes": "manual_mapping"},
    {"source_dataset": "fullcontact", "source_column": "founders", "target_dataset": "target_schema", "target_column": "founders", "score": 1.0, "notes": "manual_mapping"},
])

fullcontact_mapping

,source_dataset,source_column,target_dataset,target_column,score,notes
0,fullcontact,Attribute_1,target_schema,id,1.0,manual_mapping
1,fullcontact,Attribute_2,target_schema,name,1.0,manual_mapping
2,fullcontact,Attribute_3,target_schema,country,1.0,manual_mapping
3,fullcontact,Attribute_4,target_schema,city,1.0,manual_mapping
4,fullcontact,Attribute_6,target_schema,founded,1.0,manual_mapping
5,fullcontact,founders,target_schema,founders,1.0,manual_mapping


## Step 4: Translate and Normalize


In [11]:
translator = SchemaTranslator()
# Translate + normalize each dataset with its own mapping

# Remove , from dbpedia financials
dbpedia["total_assets_val"] = dbpedia["total_assets_val"].astype(str).str.replace(",", "", regex=False)
dbpedia["annual_income"] = dbpedia["annual_income"].astype(str).str.replace(",", "", regex=False)
for col in ["total_assets_val", "annual_income"]:
    dbpedia[col] = pd.to_numeric(dbpedia[col], errors="coerce")
    dbpedia[col] = dbpedia[col].round().astype("Int64")

spec.set_column("country", country_format="name")

forbes_normalized = translator.translate(
    forbes, forbes_mapping,
    normalize=spec, on_failure="keep"
)

dbpedia_normalized = translator.translate(
    dbpedia, dbpedia_mapping,
    normalize=spec, on_failure="keep"
)

fullcontact_normalized = translator.translate(
    fullcontact, fullcontact_mapping,
    normalize=spec, on_failure="keep"
)

In [12]:
# Inspect normalized dbpedia dataset (target columns only)
dbpedia_cols = [c for c in target_columns if c in dbpedia_normalized.columns]
dbpedia_normalized[dbpedia_cols].head(10)

,id,name,founded,country,city,industry,assets,revenue,founders
0,http://dbpedia.org/resource/%C3%80_la_Table_de...,À la Table de Spanghero,1970-01-01 00:00:00,France,Castelnaudary,Meat,<NA>,<NA>,NaN
1,http://dbpedia.org/resource/%C3%81guas_de_Port...,�?guas de Portugal,1993-01-01 00:00:00,Portugal,Lisbon,NaN,<NA>,<NA>,NaN
2,http://dbpedia.org/resource/%C3%81nima_Estudios,�?nima Estudios,2002-01-01 00:00:00,Mexico,Mexico City,Animation,<NA>,<NA>,NaN
3,http://dbpedia.org/resource/%C3%87al%C4%B1k_En...,Çalık Enerji,1998-01-01 00:00:00,Turkey,Istanbul,NaN,<NA>,<NA>,Çalık Holding
4,http://dbpedia.org/resource/%C3%87al%C4%B1k_Ho...,Çalık Holding,1997-01-01 00:00:00,Turkey,Istanbul,NaN,8,3,Ahmet Çalık
5,http://dbpedia.org/resource/%C3%87ukurova_(con...,Çukurova (construction firm),1975-01-01 00:00:00,Turkey,Istanbul,NaN,<NA>,<NA>,NaN
6,http://dbpedia.org/resource/%C3%87ukurova_Holding,Çukurova Holding,1923-01-01 00:00:00,Turkey,Istanbul,Communication,<NA>,<NA>,NaN
7,http://dbpedia.org/resource/%C3%89lectricit%C3...,Électricité de France,1946-01-01 00:00:00,France,Paris,Electric utility,240560000000,65170000000,Marcel Paul
8,http://dbpedia.org/resource/%C3%89tranges_Libe...,Étranges Libellules,1994-01-01 00:00:00,France,Lyon,Video game industry,<NA>,<NA>,NaN
9,http://dbpedia.org/resource/%C3%96ssur,Össur,1971-01-01 00:00:00,Iceland,Reykjavík,Health care,607100000,358500000,NaN


In [13]:
# Inspect normalized forbes dataset (target columns only)
forbes_cols = [c for c in target_columns if c in forbes_normalized.columns]
forbes_normalized[forbes_cols].head(10)

,id,name,country,industry,assets,revenue
0,http://www.forbes.com/companies/icbc/,ICBC,China,Major Banks,3124900000000,148700000000
1,http://www.forbes.com/companies/china-construc...,China Construction Bank,China,Regional Banks,2449500000000,121300000000
2,http://www.forbes.com/companies/agricultural-b...,Agricultural Bank of China,China,Regional Banks,2405400000000,136400000000
3,http://www.forbes.com/companies/jpmorgan-chase/,JPMorgan Chase,United States,Major Banks,2435300000000,105700000000
4,http://www.forbes.com/companies/berkshire-hath...,Berkshire Hathaway,United States,Investment Services,493400000000,178800000000
5,http://www.forbes.com/companies/exxon-mobil/,Exxon Mobil,United States,Oil & Gas Operations,346800000000,394000000000
6,http://www.forbes.com/companies/general-electric/,General Electric,United States,Conglomerates,656600000000,143300000000
7,http://www.forbes.com/companies/wells-fargo/,Wells Fargo,United States,Major Banks,1543000000000,88700000000
8,http://www.forbes.com/companies/bank-of-china/,Bank of China,China,Major Banks,2291800000000,105100000000
9,http://www.forbes.com/companies/petrochina/,PetroChina,China,Oil & Gas Operations,386900000000,328500000000


In [14]:
# Inspect normalized fullcontact dataset (target columns only)
fullcontact_cols = [c for c in target_columns if c in fullcontact_normalized.columns]
fullcontact_normalized[fullcontact_cols].head(10)

,id,name,founded,country,city,founders
0,fullcontact_1,BBMG,NaN,United States,Brooklyn,NaN
1,fullcontact_2,CIT Group Inc (DEL),1908-01-01 00:00:00,Canada,Toronto,NaN
2,fullcontact_3,City & National Employment,1957-01-01 00:00:00,United States,Waterloo,NaN
3,fullcontact_4,Ingersoll Rand South East Asia (Pte) Ltd,1871-01-01 00:00:00,Ireland,Swords,NaN
4,fullcontact_5,Evonik,2007-01-01 00:00:00,Germany,Essen,NaN
5,fullcontact_6,Exon Mobil,NaN,United States,Houston,NaN
6,fullcontact_7,PPG Industries,1883-01-01 00:00:00,United States,Pittsburgh,NaN
7,fullcontact_8,Walmart,1962-01-01 00:00:00,United States,Bentonville,NaN
8,fullcontact_9,Deutsche Börse AG,2001-01-01 00:00:00,Germany,Eschborn,NaN
9,fullcontact_10,Tabbs,NaN,NaN,NaN,NaN


In [15]:
# Only keep target columns
dbpedia = dbpedia_normalized[dbpedia_cols].copy()
forbes = forbes_normalized[forbes_cols].copy()
fullcontact = fullcontact_normalized[fullcontact_cols].copy()

## Part 2: Data Profiling

In [16]:
from PyDI.utils import DataProfiler

# Display basic information
datasets = [dbpedia, forbes, fullcontact]
names = ["DBpedia", "Forbes", "FullContact"]

total_records = sum(len(df) for df in datasets)
print(f"Total records across all datasets: {total_records:,}")

# Initialize the DataProfiler
profiler = DataProfiler()

for df, name in zip(datasets, names):
    profile = profiler.summary(df) # automatically prints some statistics and returns object containing stats

display(profile)

Total records across all datasets: 14,016
dbpedia:
  Rows: 10,085
  Columns: 9
  Total nulls: 30,513
  Null percentage: 33.6%
  Null counts per column:
    city: 700 (6.9%)
    industry: 3,207 (31.8%)
    assets: 9,425 (93.5%)
    revenue: 8,024 (79.6%)
    founders: 9,157 (90.8%)

forbes:
  Rows: 2,000
  Columns: 6
  Total nulls: 114
  Null percentage: 0.9%
  Null counts per column:
    country: 71 (3.5%)
    industry: 43 (2.1%)

fullcontact:
  Rows: 1,931
  Columns: 6
  Total nulls: 3,870
  Null percentage: 33.4%
  Null counts per column:
    founded: 875 (45.3%)
    country: 508 (26.3%)
    city: 556 (28.8%)
    founders: 1,931 (100.0%)



{'rows': 1931,
 'columns': 6,
 'nulls_total': 3870,
 'nulls_per_column': {'id': 0,
  'name': 0,
  'founded': 875,
  'country': 508,
  'city': 556,
  'founders': 1931},
 'dtypes': {'id': 'object',
  'name': 'object',
  'founded': 'object',
  'country': 'object',
  'city': 'object',
  'founders': 'float64'}}

### Attribute Coverage Analysis

In [17]:
coverage = profiler.analyze_coverage(
    datasets=datasets,
    include_samples=True,
    sample_count=3  # Show 3 sample values per attribute
)

print("📊 Attribute coverage across datasets:")
display(coverage)

# Identify attributes suitable for entity matching
print("\n🔗 Attributes suitable for entity matching:")
matching_attrs = coverage[coverage['datasets_with_attribute'] >= 2]['attribute'].tolist()
print(f"Attributes available in 2+ datasets: {matching_attrs}")

📊 Attribute coverage across datasets:


,attribute,dbpedia_count,dbpedia_pct,dbpedia_coverage,dbpedia_samples,forbes_count,forbes_pct,forbes_coverage,forbes_samples,fullcontact_count,fullcontact_pct,fullcontact_coverage,fullcontact_samples,avg_coverage,max_coverage,datasets_with_attribute
0,assets,660/10085,6.5%,0.065444,"[8, 240560000000, 607100000]",2000/2000,100.0%,1.0000,"[3124900000000, 2449500000000, 2405400000000]",0/0,0%,0.000000,N/A,0.355148,1.000000,2
1,city,9385/10085,93.1%,0.930590,"['Castelnaudary', 'Lisbon', 'Mexico City']",0/0,0%,0.0000,N/A,1375/1931,71.2%,0.712066,"['Brooklyn', 'Toronto', 'Waterloo']",0.547552,0.930590,2
2,country,10085/10085,100.0%,1.000000,"['France', 'Portugal', 'Mexico']",1929/2000,96.5%,0.9645,"['China', 'China', 'China']",1423/1931,73.7%,0.736924,"['United States', 'Canada', 'United States']",0.900475,1.000000,3
3,founded,10085/10085,100.0%,1.000000,"[Timestamp('1970-01-01 00:00:00'), Timestamp('...",0/0,0%,0.0000,N/A,1056/1931,54.7%,0.546867,"[Timestamp('1908-01-01 00:00:00'), Timestamp('...",0.515622,1.000000,2
4,founders,928/10085,9.2%,0.092018,"['Çalık Holding', 'Ahmet Çalık', 'Marcel Paul']",0/0,0%,0.0000,N/A,0/1931,0.0%,0.000000,[],0.030673,0.092018,1
5,id,10085/10085,100.0%,1.000000,['http://dbpedia.org/resource/%C3%80_la_Table_...,2000/2000,100.0%,1.0000,"['http://www.forbes.com/companies/icbc/', 'htt...",1931/1931,100.0%,1.000000,"['fullcontact_1', 'fullcontact_2', 'fullcontac...",1.000000,1.000000,3
6,industry,6878/10085,68.2%,0.682003,"['Meat', 'Animation', 'Communication']",1957/2000,97.9%,0.9785,"['Major Banks', 'Regional Banks', 'Regional Ba...",0/0,0%,0.000000,N/A,0.553501,0.978500,2
7,name,10085/10085,100.0%,1.000000,"['À la Table de Spanghero', '�?guas de Portuga...",2000/2000,100.0%,1.0000,"['ICBC', 'China Construction Bank', 'Agricultu...",1931/1931,100.0%,1.000000,"['BBMG', 'CIT Group Inc (DEL)', 'City & Nation...",1.000000,1.000000,3
8,revenue,2061/10085,20.4%,0.204363,"[3, 65170000000, 358500000]",2000/2000,100.0%,1.0000,"[148700000000, 121300000000, 136400000000]",0/0,0%,0.000000,N/A,0.401454,1.000000,2



🔗 Attributes suitable for entity matching:
Attributes available in 2+ datasets: ['assets', 'city', 'country', 'founded', 'id', 'industry', 'name', 'revenue']


## Part 3: Entity Matching

### Step 1: Blocking

In [18]:
# Set up logging
import logging

import os
os.makedirs('output/logs', exist_ok=True)

logging.basicConfig(
    level=logging.INFO, # Alternatively, use logging.DEBUG for more verbosity
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler('output/logs/pydi.log'),  # Save to file
          logging.StreamHandler()                      # Display on console
      ],
    force=True
)

In [19]:
from PyDI.entitymatching import TokenBlocker

token_blocker_f2d = TokenBlocker(
    forbes, dbpedia,
    column='name',
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)

token_blocker_f2fc = TokenBlocker(
    forbes, fullcontact,
    column='name',
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)

[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 2295 token keys for first dataset
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 11043 token keys for second dataset
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 1111 blocks from token keys
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - Debug results written to file: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/companies/output/blocking-evaluation/debugResultsBlocking_TokenBlocker.csv
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 2295 token keys for first dataset
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 2709 token keys for second dataset
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 1122 blocks from token keys
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - Debug results written to file: /Users/aaronsteiner/Documen

### Step 2: Evaluate Blocking Against Ground Truth

In [20]:
import pandas as pd
from PyDI.io import load_csv
from PyDI.entitymatching import EntityMatchingEvaluator
# Showcase EntityMatchingEvaluator.evaluate_blocking utility

# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "forbes_2_dbpedia_val.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking_batched on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=token_blocker_f2d,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

[INFO ] root - Starting batched blocking evaluation...
[INFO ] root - Processed 10 batches, 10000 pairs, 4 true matches
[INFO ] root - Processed 20 batches, 20000 pairs, 7 true matches
[INFO ] root - Processed 30 batches, 30000 pairs, 9 true matches
[INFO ] root - Processed 40 batches, 40000 pairs, 11 true matches
[INFO ] root - Processed 50 batches, 50000 pairs, 12 true matches
[INFO ] root - Processed 60 batches, 60000 pairs, 12 true matches
[INFO ] root - Processed 70 batches, 70000 pairs, 12 true matches
[INFO ] root - Processed 80 batches, 80000 pairs, 12 true matches
[INFO ] root - Processed 90 batches, 90000 pairs, 14 true matches
[INFO ] root - Processed 100 batches, 100000 pairs, 22 true matches
[INFO ] root - Processed 110 batches, 110000 pairs, 27 true matches
[INFO ] root - Processed 120 batches, 120000 pairs, 42 true matches
[INFO ] root - Processed 130 batches, 129861 pairs, 101 true matches
[INFO ] root -   Pair Completeness: 0.971
[INFO ] root -   Pair Quality:      0.0

{'pair_completeness': 0.9711538461538461,
 'pair_quality': 0.0007777546761537336,
 'reduction_ratio': 0.9935616757560733,
 'total_candidates': 129861,
 'total_possible_pairs': 20170000,
 'true_positives_found': 101,
 'total_true_pairs': 104,
 'batches_processed': 130,
 'evaluation_timestamp': '2026-02-04T18:39:16.213221',
 'output_files': ['/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/companies/output/blocking-evaluation/blocking_evaluation_summary.json',
  '/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/companies/output/blocking-evaluation/blocking_detailed_results.csv']}

In [21]:
# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "forbes_2_fullcontact_val.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking_batched on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=token_blocker_f2fc,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

[INFO ] root - Starting batched blocking evaluation...


[INFO ] root - Processed 10 batches, 10000 pairs, 12 true matches
[INFO ] root - Processed 20 batches, 20000 pairs, 15 true matches
[INFO ] root - Processed 30 batches, 29500 pairs, 218 true matches
[INFO ] root -   Pair Completeness: 0.944
[INFO ] root -   Pair Quality:      0.007
[INFO ] root -   Reduction Ratio:   0.992361
[INFO ] root -   True Matches Found: 218/231
[INFO ] root -   Batches Processed:  30
[INFO ] root - Blocking evaluation complete!


{'pair_completeness': 0.9437229437229437,
 'pair_quality': 0.007389830508474577,
 'reduction_ratio': 0.9923614707405489,
 'total_candidates': 29500,
 'total_possible_pairs': 3862000,
 'true_positives_found': 218,
 'total_true_pairs': 231,
 'batches_processed': 30,
 'evaluation_timestamp': '2026-02-04T18:39:17.700872',
 'output_files': ['/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/companies/output/blocking-evaluation/blocking_evaluation_summary.json',
  '/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/companies/output/blocking-evaluation/blocking_detailed_results.csv']}

### Step 3: Entity Matching with Comparators

In [22]:
from PyDI.entitymatching import StringComparator
import re

# ignore case and punctuation
def normalize_text(s: str) -> str: 
    if s is None:
        return ""
    return re.sub(r"[^\w\s]|_", "", s).lower()

comparators_f2d = [
    StringComparator(
        column='name', 
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    StringComparator(
        column='name', 
        similarity_function='levenshtein',
        preprocess=normalize_text
    ),
    StringComparator(
        column='country',
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    StringComparator(
        column='industry',
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
]

comparators_f2fc = [
    StringComparator(
        column='name', 
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    StringComparator(
        column='country',
        similarity_function='jaccard',
        preprocess=normalize_text
    )
]

Next, we setup the matcher and run the matching with our chosen best blocking method:

In [23]:
from PyDI.entitymatching import RuleBasedMatcher

# Initialize Rule-Based Matcher
matcher = RuleBasedMatcher()

correspondences_f2d = matcher.match(
    df_left=forbes,
    df_right=dbpedia, 
    candidates=token_blocker_f2d,
    comparators=comparators_f2d,
    weights=[1.0, 1.0, 1.0, 0.3],
    threshold=0.2,
    id_column='id'
)

correspondences_f2fc = matcher.match(
    df_left=forbes,
    df_right=fullcontact, 
    candidates=token_blocker_f2fc,
    comparators=comparators_f2fc,
    weights=[1.0, 0.5],
    threshold=0.1,
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 2000 x 10085 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 2000 x 10085 elements after 0:00:0.039; 129861 blocked pairs (reduction ratio: 0.9935616757560733)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:11.711; found 59240 correspondences.
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 2000 x 1931 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 2000 x 1931 elements after 0:00:0.011; 29500 blocked pairs (reduction ratio: 0.9923614707405489)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:2.209; found 27073 correspondences.


### Step 4: Evaluate Matching Against Ground Truth

In [24]:
gt_val = load_csv(
    INPUT_DIR / "entitymatching" / "forbes_2_dbpedia_val.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_f2d,
    test_pairs=gt_val,
    out_dir=debug_output_dir
)

display(eval_results)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  101
[INFO ] root -   True Negatives:  60
[INFO ] root -   False Positives: 55
[INFO ] root -   False Negatives: 3
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.735
[INFO ] root -   Precision: 0.647
[INFO ] root -   Recall:    0.971
[INFO ] root -   F1-Score:  0.777


{'precision': 0.6474358974358975,
 'recall': 0.9711538461538461,
 'f1': 0.7769230769230769,
 'accuracy': 0.7351598173515982,
 'true_positives': 101,
 'false_positives': 55,
 'false_negatives': 3,
 'true_negatives': 60,
 'threshold_used': 0.0,
 'total_correspondences': 59240,
 'filtered_correspondences': 59240,
 'evaluation_timestamp': '2026-02-04T18:39:32.587686',
 'output_files': ['/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/companies/output/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/companies/output/debug_results_entity_matching/matching_detailed_results.csv']}

In [25]:
print("Analyzing cluster size distribution in our entity matching results...")

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_f2d,
    out_dir=str(OUTPUT_DIR / "cluster_analysis")
)

print(f"\n📊 Cluster Size Distribution Results:")
display(cluster_distribution)

Analyzing cluster size distribution in our entity matching results...


[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 184 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	145	|	78.80%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	18	|	9.78%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	8	|	4.35%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	3	|	1.63%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	2	|	1.09%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	4	|	2.17%
[INFO ] PyDI.entitymatching.evaluation - 		8	|	1	|	0.54%
[INFO ] PyDI.entitymatching.evaluation - 		11	|	1	|	0.54%
[INFO ] PyDI.entitymatching.evaluation - 		22	|	1	|	0.54%
[INFO ] PyDI.entitymatching.evaluation - 		5347	|	1	|	0.54%
[INFO ] root - Cluster size distribution written to /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/companies/output/cluster_analysis/cluster_size_distribu


📊 Cluster Size Distribution Results:


,cluster_size,frequency,percentage
0,2,145,78.804348
1,3,18,9.782609
2,4,8,4.347826
3,5,3,1.630435
4,6,2,1.086957
5,7,4,2.173913
6,8,1,0.543478
7,11,1,0.543478
8,22,1,0.543478
9,5347,1,0.543478


In [26]:
# Write out detailed cluster information with all entity records for debugging purposes

# Use the matches we found earlier to demonstrate cluster details
cluster_details_path = OUTPUT_DIR / "cluster_analysis" / "detailed_cluster_info.json"

# Call write_cluster_details with our entity matches
output_path = EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_f2d,
    out_path=cluster_details_path
)

[INFO ] root - Cluster details written to /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/companies/output/cluster_analysis/detailed_cluster_info.json
[INFO ] root - Exported 184 clusters with detailed record information


Additionally, PyDI offers 6 different post-clustering methods to "clean" clusters after entity matching. For example, if we want to enforce that each record in a dataset can only have exactly one correspondence in the other dataset, we can apply a greedy one-to-one matching, maximum bipartite matching or stable marriage matching.

In [27]:
from PyDI.entitymatching import GreedyOneToOneMatchingAlgorithm

# use Greedy One-To-One Matching to refine results to 1:1 matches
clusterer = GreedyOneToOneMatchingAlgorithm()
correspondences_f2d = clusterer.cluster(correspondences_f2d)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_f2d,
    test_pairs=gt_val,
    out_dir=debug_output_dir
)

display(eval_results)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_f2d,
    out_dir=str(OUTPUT_DIR / "cluster_analysis")
)

print(f"\n📊 Cluster Size Distribution Results:")
display(cluster_distribution)

[INFO ] root - Filtered correspondences: 59240 -> 59240 (threshold=0.0)
[INFO ] root - Greedy matching: 59240 -> 1404 correspondences (2808 entities matched)
[INFO ] root - GreedyOneToOneMatchingAlgorithm: 59240 -> 1404 correspondences
[INFO ] root - GreedyOneToOneMatchingAlgorithm: 5819 -> 2808 entities
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  93
[INFO ] root -   True Negatives:  109
[INFO ] root -   False Positives: 6
[INFO ] root -   False Negatives: 11
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.922
[INFO ] root -   Precision: 0.939
[INFO ] root -   Recall:    0.894
[INFO ] root -   F1-Score:  0.916


{'precision': 0.9393939393939394,
 'recall': 0.8942307692307693,
 'f1': 0.9162561576354681,
 'accuracy': 0.9223744292237442,
 'true_positives': 93,
 'false_positives': 6,
 'false_negatives': 11,
 'true_negatives': 109,
 'threshold_used': 0.0,
 'total_correspondences': 1404,
 'filtered_correspondences': 1404,
 'evaluation_timestamp': '2026-02-04T18:39:36.773780',
 'output_files': ['/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/companies/output/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/companies/output/debug_results_entity_matching/matching_detailed_results.csv']}

[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 1404 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	1404	|	100.00%
[INFO ] root - Cluster size distribution written to /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/companies/output/cluster_analysis/cluster_size_distribution.csv



📊 Cluster Size Distribution Results:


,cluster_size,frequency,percentage
0,2,1404,100.0


In [28]:
gt_val = load_csv(
    INPUT_DIR / "entitymatching" / "forbes_2_fullcontact_val.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_f2fc,
    test_pairs=gt_val,
    out_dir=debug_output_dir
)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_f2fc,
)

clusterer = GreedyOneToOneMatchingAlgorithm()
correspondences_f2fc = clusterer.cluster(correspondences_f2fc)


cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_f2fc,
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_f2fc,
    test_pairs=gt_val,
    out_dir=debug_output_dir
)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  217
[INFO ] root -   True Negatives:  406
[INFO ] root -   False Positives: 92
[INFO ] root -   False Negatives: 14
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.855
[INFO ] root -   Precision: 0.702
[INFO ] root -   Recall:    0.939
[INFO ] root -   F1-Score:  0.804
[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 285 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	266	|	93.33%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	12	|	4.21%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	3	|	1.05%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	1	|	0.35%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	1	|	0.35%
[INFO ] PyDI.entitymatching.evaluation - 		11	|	1	|	0.35%
[INFO ] PyDI.entitymatching.evaluation - 

Now, we can directly use the trained model with PyDIs MLBasedMatcher

## Part 4: Data Fusion

In [29]:
forbes["forbes_id"] = forbes["id"]

# Assign trust scores to datasets
forbes.attrs["trust_score"] = 1
dbpedia.attrs["trust_score"] = 3
fullcontact.attrs["trust_score"] = 2

all_correspondences = pd.concat([correspondences_f2d, correspondences_f2fc], ignore_index=True)
print(f'Total correspondences: {len(all_correspondences):,}')

Total correspondences: 2,499


## Step 1: Define Fusion Strategy

In [30]:
from PyDI.fusion import DataFusionStrategy, longest_string, shortest_string, union, prefer_higher_trust, voting, maximum, most_recent

strategy = DataFusionStrategy('company_fusion_strategy')

strategy.add_attribute_fuser('name', voting)
strategy.add_attribute_fuser('assets', prefer_higher_trust)
strategy.add_attribute_fuser('revenue', prefer_higher_trust)
strategy.add_attribute_fuser('founders', union)
strategy.add_attribute_fuser('founded', prefer_higher_trust)
strategy.add_attribute_fuser('country', voting)
strategy.add_attribute_fuser('city', shortest_string)

[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'name' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'assets' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'revenue' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'founders' using rule 'union'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'founded' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'country' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'city' using rule 'shortest_string'


## Step 2: Run Fusion

In [31]:
from PyDI.fusion import DataFusionEngine

engine = DataFusionEngine(strategy, debug=True, debug_format='json',debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion.jsonl")

fused = engine.run(
    datasets=[forbes, dbpedia, fullcontact],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=True,
)
print(f'Fused rows: {len(fused):,}')
display(fused.head(5))

[INFO ] PyDI.fusion.engine - Fusion debug logging enabled; refer to /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/companies/output/data_fusion/debug_fusion.jsonl for detailed traces.
[INFO ] PyDI.fusion.engine - Starting data fusion with strategy 'company_fusion_strategy'
[INFO ] PyDI.fusion.engine - *    Loading correspondences    *
[INFO ] PyDI.fusion.engine - Correspondence ID coverage: matched 4236 of 4236 unique IDs
[INFO ] PyDI.fusion.engine - Created 11517 record groups from 2499 correspondences
[INFO ] PyDI.fusion.engine - Group Size Distribution of 11517 clusters:
[INFO ] PyDI.fusion.engine - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.fusion.engine - 	──────────────────────────────────────────────────
[INFO ] PyDI.fusion.engine - 		2	|	975	|	8.47%
[INFO ] PyDI.fusion.engine - 		3	|	762	|	6.62%
[INFO ] PyDI.fusion.engine - Attribute Consistencies:
[INFO ] PyDI.fusion.engine -     _id: 0.00
[INFO ] PyDI.fusion.engine -     assets: 0.85
[INFO ] PyDI.fusion.eng

Fused rows: 11,517


,_id,_fusion_sources,_fusion_source_datasets,assets,forbes_id,id,industry,founded,name,country,revenue,founders,city,_fusion_confidence,_fusion_metadata
0,http://www.forbes.com/companies/israel-discoun...,[http://www.forbes.com/companies/israel-discou...,"[forbes, dbpedia, fullcontact]",5.780000e+10,http://www.forbes.com/companies/israel-discoun...,http://www.forbes.com/companies/israel-discoun...,Regional Banks,1935-01-01 00:00:00,Israel Discount Bank,Israel,4.000000e+06,None,Tel Aviv (Bank Discount Tower),0.740741,"{'assets_rule': 'prefer_higher_trust', 'assets..."
1,fullcontact_512,"[fullcontact_512, http://dbpedia.org/resource/...","[fullcontact, dbpedia, forbes]",1.424300e+11,http://www.forbes.com/companies/microsoft/,fullcontact_512,Computer hardware,1975-01-01 00:00:00,Microsoft,United States,7.785000e+10,"[['Paul Allen', 'Bill Gates']]",Redmond,0.879845,"{'assets_rule': 'prefer_higher_trust', 'assets..."
2,http://www.forbes.com/companies/prudential/,"[http://www.forbes.com/companies/prudential/, ...","[forbes, fullcontact, dbpedia]",5.285000e+11,http://www.forbes.com/companies/prudential/,http://www.forbes.com/companies/prudential/,Life & Health Insurance,1848-01-01 00:00:00,Prudential,United Kingdom,3.050200e+10,None,London,0.629630,"{'assets_rule': 'prefer_higher_trust', 'assets..."
3,http://dbpedia.org/resource/Exel_NA_Industrial,[http://dbpedia.org/resource/Exel_NA_Industria...,"[dbpedia, forbes]",1.020000e+10,http://www.forbes.com/companies/daelim-industr...,http://dbpedia.org/resource/Exel_NA_Industrial,Construction Services,1925-01-01 00:00:00,Exel NA Industrial,United States,9.000000e+09,None,PlymouthMichigan,0.666667,"{'assets_rule': 'prefer_higher_trust', 'assets..."
4,http://dbpedia.org/resource/Singapore_Post,"[http://dbpedia.org/resource/Singapore_Post, f...","[dbpedia, fullcontact, forbes]",1.790000e+10,http://www.forbes.com/companies/singapore-airl...,http://dbpedia.org/resource/Singapore_Post,Conglomerate (company),1819-01-01 00:00:00,Singapore Airlines,Singapore,1.220000e+10,None,Singapore,0.740741,"{'assets_rule': 'prefer_higher_trust', 'assets..."


## Step 3: Evaluate Data Fusion

In [32]:
from PyDI.fusion import tokenized_match, year_only_match, set_equality_match, numeric_tolerance_match

strategy.add_evaluation_function("name", tokenized_match)
strategy.add_evaluation_function("assets", tokenized_match)
strategy.add_evaluation_function("revenue", numeric_tolerance_match, tolerance=0.1)
strategy.add_evaluation_function("assets", numeric_tolerance_match, tolerance=0.1)
strategy.add_evaluation_function("founders", set_equality_match)
strategy.add_evaluation_function("founded", year_only_match)
strategy.add_evaluation_function("country", tokenized_match)
strategy.add_evaluation_function("city", tokenized_match)

[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'name'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'assets'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'revenue' with params {'tolerance': 0.1}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'assets' with params {'tolerance': 0.1}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'founders'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'founded'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'country'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'city'


In [33]:
from PyDI.fusion import DataFusionEvaluator

fusion_test_set = load_xml(INPUT_DIR / 'fusion' / 'test_set.xml', name='fusion_test_set', nested_handling='aggregate')
# rename keypeople_name to founders for evaluation
fusion_test_set['founders'] = fusion_test_set['keypeople_name'].apply(lambda x: [x] if isinstance(x, str) else x)

# convert scientific number notation into integers
def convert_scientific_notation(value):
    try:
        if isinstance(value, str) and ('e' in value or 'E' in value):
            return int(float(value))
        return value
    except:
        return value

fusion_test_set['assets'] = fusion_test_set['assets'].apply(convert_scientific_notation)
fusion_test_set['revenue'] = fusion_test_set['revenue'].apply(convert_scientific_notation) 

# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the gold standard
print("Evaluating fusion results against gold standard...")
evaluation_results = evaluator.evaluate(
    fused_df=fused,
    fused_id_column='forbes_id',
    gold_df=fusion_test_set,
    gold_id_column='id',
)

# Display evaluation metrics
print("\nFusion Evaluation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

[INFO ] PyDI.fusion.evaluation - Fusion evaluation debug logging enabled; refer to /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/companies/output/data_fusion/debug_fusion_eval.jsonl for mismatch details.
[INFO ] PyDI.fusion.evaluation - Starting fusion evaluation
[INFO ] PyDI.fusion.evaluation - Evaluation complete: 0.803 overall accuracy (94/117)
[INFO ] PyDI.fusion.evaluation - Evaluation mismatches by attribute (debug): 23 total
[INFO ] PyDI.fusion.evaluation - 	Attribute                        |  Errors | Percentage
[INFO ] PyDI.fusion.evaluation - 	───────────────────────────────────────────────────────
[INFO ] PyDI.fusion.evaluation - 	founders                         |       7 |     30.43%%
[INFO ] PyDI.fusion.evaluation - 	revenue                          |       5 |     21.74%%
[INFO ] PyDI.fusion.evaluation - 	city                             |       4 |     17.39%%
[INFO ] PyDI.fusion.evaluation - 	founded                          |       2 |      8.70%%
[INFO ] P

Evaluating fusion results against gold standard...

Fusion Evaluation Results:
  overall_accuracy: 0.803
  macro_accuracy: 0.762
  num_evaluated_records: 18
  num_evaluated_attributes: 7
  total_evaluations: 117
  total_correct: 94
  assets_accuracy: 0.944
  assets_count: 18
  founded_accuracy: 0.889
  founded_count: 18
  name_accuracy: 0.889
  name_count: 18
  country_accuracy: 0.889
  country_count: 18
  revenue_accuracy: 0.722
  revenue_count: 18
  founders_accuracy: 0.222
  founders_count: 9
  city_accuracy: 0.778
  city_count: 18

Overall Accuracy: 80.3%


## Part 5: Generate Reports

Generate pipeline-style reports for this workflow.

In [34]:
from PyDI.pipeline.reporting import (
    save_source_overview,
    save_schema_matching_report,
    generate_column_mapping_table,
    find_example_records,
)

# Create reporting directory
REPORTING_DIR = OUTPUT_DIR / "reporting"
REPORTING_DIR.mkdir(parents=True, exist_ok=True)

In [35]:
# Generate source overview report
sources = {
    "dbpedia": dbpedia,
    "forbes": forbes,
    "fullcontact": fullcontact,
}

# Number of target columns
num_target_cols = len(target_columns)

source_overview_path = save_source_overview(
    sources=sources,
    output_dir=REPORTING_DIR,
    target_columns=num_target_cols,
)
print(f"Saved source overview to: {source_overview_path}")

# Display the report
source_overview = pd.read_csv(source_overview_path)
display(source_overview)

Saved source overview to: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/companies/output/reporting/source_overview.csv


,Source,Rows,Columns,Data Density,Density in Target Schema (10 cols)
0,Dbpedia,10085,9,66.38%,59.74% (9/10 × 66.38%)
1,Forbes,2000,7,99.19%,69.43% (7/10 × 99.19%)
2,Fullcontact,1931,6,66.60%,39.96% (6/10 × 66.60%)
3,Total Input,14016,10 (target),77.39% avg,58.40% weighted avg


In [36]:
# Generate schema matching report
mappings = {
    "dbpedia": dbpedia_mapping,
    "forbes": forbes_mapping,
    "fullcontact": fullcontact_mapping,
}

# Extract target schema properties from $defs.Company for the report
target_schema_for_report = {"properties": target_schema["$defs"]["Company"]["properties"]}

# Create schema_matching subdirectory
schema_matching_dir = REPORTING_DIR / "schema_matching"
schema_matching_dir.mkdir(parents=True, exist_ok=True)

# Get correspondences as tuples for finding example records
correspondence_tuples = list(zip(
    all_correspondences["id1"].tolist(),
    all_correspondences["id2"].tolist()
))

# Save schema matching report (column_mapping.csv and example_records.csv)
mapping_csv_path, _ = save_schema_matching_report(
    mappings=mappings,
    target_schema=target_schema_for_report,
    output_dir=schema_matching_dir,
    normalized_sources=sources,
    correspondences=correspondence_tuples,
)
print(f"Saved column mapping to: {mapping_csv_path}")

# Display column mapping
column_mapping = pd.read_csv(mapping_csv_path)
display(column_mapping)

Saved column mapping to: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/companies/output/reporting/schema_matching/column_mapping.csv


,Target Column,Dbpedia,Forbes,Fullcontact
0,id,entity_uri,forbes_url,Attribute_1
1,name,org_name,company,Attribute_2
2,website,-,-,-
3,founded,established,-,Attribute_6
4,country,nation,region,Attribute_3
5,city,headquarters,-,Attribute_4
6,industry,sector,business_segment,-
7,assets,total_assets_val,asset_value,-
8,revenue,annual_income,sales_figure,-
9,founders,founders,-,founders


In [37]:
# Display example records (same company across sources)
example_records_path = schema_matching_dir / "example_records.csv"
if example_records_path.exists():
    example_records = pd.read_csv(example_records_path)
    print("Example records (same entity across sources):")
    display(example_records)
else:
    print("No example records found")

Example records (same entity across sources):


,id,name,founded,country,city,industry,assets,revenue,founders,Source,forbes_id
0,http://dbpedia.org/resource/Shui_On_Land,Shui On Land,2004-01-01,China,Shanghai,Real estate,NaN,NaN,NaN,Dbpedia,NaN
1,http://www.forbes.com/companies/shui-on-land/,Shui On Land,NaN,China,NaN,Real Estate,1.630000e+10,1.700000e+09,NaN,Forbes,http://www.forbes.com/companies/shui-on-land/
2,fullcontact_1187,Game on,NaN,United States,Bradenton,NaN,NaN,NaN,NaN,Fullcontact,NaN


In [38]:
# Generate Entity Matching Summary Report
entity_matching_summary = pd.DataFrame([
    {
        "left": "forbes",
        "right": "dbpedia",
        "matcher": "RuleBasedMatcher + GreedyOneToOne",
        "precision": 0.939,  # From eval_results above
        "recall": 0.894,
        "f1": 0.916,
        "correspondences": len(correspondences_f2d),
    },
    {
        "left": "forbes",
        "right": "fullcontact",
        "matcher": "RuleBasedMatcher + GreedyOneToOne",
        "precision": 0.985,  # From eval_results above
        "recall": 0.861,
        "f1": 0.919,
        "correspondences": len(correspondences_f2fc),
    },
])

# Save to reporting directory
entity_matching_path = REPORTING_DIR / "entity_matching_summary.csv"
entity_matching_summary.to_csv(entity_matching_path, index=False)
print(f"Saved entity matching summary to: {entity_matching_path}")
display(entity_matching_summary)

Saved entity matching summary to: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/companies/output/reporting/entity_matching_summary.csv


,left,right,matcher,precision,recall,f1,correspondences
0,forbes,dbpedia,RuleBasedMatcher + GreedyOneToOne,0.939,0.894,0.916,1404
1,forbes,fullcontact,RuleBasedMatcher + GreedyOneToOne,0.985,0.861,0.919,1095


In [39]:
# Generate Fusion Evaluation Summary Report
# Extract per-attribute accuracy from evaluation_results
fusion_eval_summary = {
    "overall_accuracy": evaluation_results.get("overall_accuracy", 0),
    "total_evaluated": evaluation_results.get("total_evaluations", 0),
    "total_correct": evaluation_results.get("total_correct", 0),
}

# Add per-attribute accuracies
attr_accuracies = {}
for key, value in evaluation_results.items():
    if key.endswith("_accuracy") and key != "overall_accuracy" and key != "macro_accuracy":
        attr_name = key.replace("_accuracy", "")
        attr_accuracies[attr_name] = value

fusion_eval_df = pd.DataFrame([
    {"attribute": attr, "accuracy": acc}
    for attr, acc in attr_accuracies.items()
])
fusion_eval_df = fusion_eval_df.sort_values("accuracy", ascending=False)

# Save to reporting/fusion directory
fusion_report_dir = REPORTING_DIR / "fusion"
fusion_report_dir.mkdir(parents=True, exist_ok=True)
fusion_eval_path = fusion_report_dir / "fusion_evaluation_summary.csv"
fusion_eval_df.to_csv(fusion_eval_path, index=False)
print(f"Saved fusion evaluation to: {fusion_eval_path}")
print(f"\nOverall Accuracy: {fusion_eval_summary['overall_accuracy']:.1%}")
print(f"Total Evaluations: {fusion_eval_summary['total_evaluated']} ({fusion_eval_summary['total_correct']} correct)")
print("\nPer-Attribute Accuracy:")
display(fusion_eval_df)

Saved fusion evaluation to: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/companies/output/reporting/fusion/fusion_evaluation_summary.csv

Overall Accuracy: 80.3%
Total Evaluations: 117 (94 correct)

Per-Attribute Accuracy:


,attribute,accuracy
0,assets,0.944444
1,founded,0.888889
2,name,0.888889
3,country,0.888889
6,city,0.777778
4,revenue,0.722222
5,founders,0.222222


In [40]:
# Generate End-to-End Integration Metrics
from PyDI.pipeline.end_to_end_metrics import (
    calculate_density,
    EndToEndMetrics,
    SourceStats,
    save_end_to_end_report,
    generate_end_to_end_report,
)

# Calculate per-source statistics
per_source_stats = []
for name, df in sources.items():
    density = calculate_density(df)
    cols = [c for c in df.columns if not c.startswith("_")]
    per_source_stats.append(SourceStats(
        name=name,
        rows=len(df),
        columns=len(cols),
        density=density,
    ))

# Calculate merged record count (records from 2+ sources)
merged_count = 0
if "_fusion_source_datasets" in fused.columns:
    for val in fused["_fusion_source_datasets"]:
        if isinstance(val, list) and len(val) > 1:
            merged_count += 1
        elif isinstance(val, str) and val not in ("", "[]"):
            try:
                parsed = json.loads(val)
                if isinstance(parsed, list) and len(parsed) > 1:
                    merged_count += 1
            except:
                pass

# Calculate metrics
total_input_rows = sum(s.rows for s in per_source_stats)
max_source = max(per_source_stats, key=lambda s: s.rows)
fused_density = calculate_density(fused)
fused_cols = [c for c in fused.columns if not c.startswith("_fusion_")]
avg_input_density = sum(s.density * s.rows for s in per_source_stats) / total_input_rows

end_to_end_metrics = EndToEndMetrics(
    num_sources=len(sources),
    total_input_rows=total_input_rows,
    total_input_columns=len(target_columns),
    avg_input_density=avg_input_density,
    per_source_stats=per_source_stats,
    fused_rows=len(fused),
    fused_columns=len(fused_cols),
    fused_density=fused_density,
    max_source_rows=max_source.rows,
    row_gain_over_largest=len(fused) - max_source.rows,
    row_gain_pct=((len(fused) - max_source.rows) / max_source.rows * 100) if max_source.rows > 0 else 0,
    largest_source_name=max_source.name,
    largest_source_density=max_source.density,
    merged_records=merged_count,
    density_change=fused_density - avg_input_density,
)

# Save reports
txt_path, csv_path = save_end_to_end_report(
    metrics=end_to_end_metrics,
    output_dir=REPORTING_DIR,
)

# Also save the JSON metrics
metrics_json_path = REPORTING_DIR / "end_to_end_metrics.json"
end_to_end_metrics.save(metrics_json_path)

print(f"Saved end-to-end report to: {txt_path}")
print(f"Saved end-to-end CSV to: {csv_path}")
print(f"Saved end-to-end JSON to: {metrics_json_path}")

# Print the report
print("\n" + generate_end_to_end_report(end_to_end_metrics))

# Display summary table
display(end_to_end_metrics.summary_table())

Saved end-to-end report to: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/companies/output/reporting/end_to_end_report.txt
Saved end-to-end CSV to: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/companies/output/reporting/end_to_end_report.csv
Saved end-to-end JSON to: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/companies/output/reporting/end_to_end_metrics.json

END-TO-END INTEGRATION METRICS

Input Sources:                      3
Total Input Records:                14,016
Fused Output Records:               11,517
Fusion Ratio:                       82.2%

Row Gain Over Largest Source:       +1,432
Row Gain Percentage:                +14.2%
Merged Records:                     1,737

Input Columns (Target Schema):      10
Fused Output Columns:               11

Average Input Data Density:         71.1%
Fused Data Density:                 65.8%
Data Density Change:                -5.2%

Largest Source:                     dbpedia
Largest Source Density

,Metric,Value
0,Input Sources,3
1,Total Input Records,"14,016"
2,Fused Output Records,"11,517"
3,Fusion Ratio,82.2%
4,Row Gain Over Largest Source,"+1,432"
5,Row Gain Percentage,+14.2%
6,Merged Records,"1,737"
7,Input Columns (Target Schema),10
8,Fused Output Columns,11
9,Average Input Data Density,71.1%


In [41]:
# Summary of generated reports
print("=" * 60)
print("GENERATED REPORTS")
print("=" * 60)

import os
for root, dirs, files in os.walk(REPORTING_DIR):
    level = root.replace(str(REPORTING_DIR), '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = '  ' * (level + 1)
    for file in files:
        filepath = Path(root) / file
        size = filepath.stat().st_size
        print(f"{subindent}{file} ({size:,} bytes)")

GENERATED REPORTS
reporting/
  end_to_end_report.txt (845 bytes)
  source_overview.csv (271 bytes)
  end_to_end_report.csv (407 bytes)
  entity_matching_summary.csv (203 bytes)
  end_to_end_metrics.json (822 bytes)
  fusion/
    fusion_evaluation_summary.csv (202 bytes)
  schema_matching/
    example_records.csv (410 bytes)
    column_mapping.csv (387 bytes)
